In [1]:
#Abertura das Bibliotecas utilizadas
import pandas as pd
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import netCDF4 as nc
import plotly.express as px
import plotly.graph_objects as go
import cartopy.crs as ccrs # Escolha do sistema de coordenadas
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
import cartopy.feature as cfeature
import matplotlib.patches as mpatches # Desenhar geometria em um mapa
import matplotlib.ticker as mticker
import rasterio

In [7]:
# Colocar no nome do arquivo que deseja carregar
arquivo = "Precip.tif"

In [8]:
import rasterio

#Abre o arquivo de Tmax tif.
Precip_GPM = rasterio.open(arquivo).read()
print('precip_GPM',Precip_GPM.shape)

precip_GPM (318, 750, 551)


In [3]:
# Separa os valores do shape
mesesGPM, latitudeGPM, longitudeGPM = Precip_GPM.shape

# Imprime os valores separados
print('Número de dados de tempo:', mesesGPM)
print('Número de latitudes:', latitudeGPM)
print('Número de longitudes:', longitudeGPM)

Número de dados de tempo: 318
Número de latitudes: 750
Número de longitudes: 551


In [10]:
import numpy as np

#Definição da data inicial
data_inicial=np.datetime64('1998-01', 'M')

#Criação de uma lista com as datas mensais - levando em consideração o número de linha de tempo presentes nos dados
data_GPM = np.arange(data_inicial,data_inicial+mesesGPM, dtype="datetime64[M]")

data_GPM = np.array(data_GPM, dtype="datetime64[ns]") 
print("Dados de tempo:",data_GPM.shape)
data_GPM[:-10]

Dados de tempo: (318,)


array(['1998-01-01T00:00:00.000000000', '1998-02-01T00:00:00.000000000',
       '1998-03-01T00:00:00.000000000', '1998-04-01T00:00:00.000000000',
       '1998-05-01T00:00:00.000000000', '1998-06-01T00:00:00.000000000',
       '1998-07-01T00:00:00.000000000', '1998-08-01T00:00:00.000000000',
       '1998-09-01T00:00:00.000000000', '1998-10-01T00:00:00.000000000',
       '1998-11-01T00:00:00.000000000', '1998-12-01T00:00:00.000000000',
       '1999-01-01T00:00:00.000000000', '1999-02-01T00:00:00.000000000',
       '1999-03-01T00:00:00.000000000', '1999-04-01T00:00:00.000000000',
       '1999-05-01T00:00:00.000000000', '1999-06-01T00:00:00.000000000',
       '1999-07-01T00:00:00.000000000', '1999-08-01T00:00:00.000000000',
       '1999-09-01T00:00:00.000000000', '1999-10-01T00:00:00.000000000',
       '1999-11-01T00:00:00.000000000', '1999-12-01T00:00:00.000000000',
       '2000-01-01T00:00:00.000000000', '2000-02-01T00:00:00.000000000',
       '2000-03-01T00:00:00.000000000', '2000-04-01

In [9]:
import rasterio
import numpy as np

# Abrir o arquivo .tif
with rasterio.open(arquivo) as src:
    # Ler os dados da imagem (3D - Tempo, Latitude, Longitude)
    precip_data = src.read()

    # Obter a transformação espacial (informações de georreferenciamento)
    transform = src.transform

    # Obter os limites geográficos do arquivo
    bounds = src.bounds

    # Obter as dimensões da imagem (dimensões lat, lon)
    height = src.height
    width = src.width

    # Obter as coordenadas de latitude e longitude
    # O arquivo tem 3 dimensões (tempo, latitude, longitude), então vamos pegar as latitudes e longitudes
    latitudes_GPM = np.linspace(bounds[3], bounds[1], height)  # De baixo para cima
    longitudes_GPM = np.linspace(bounds[0], bounds[2], width)  # Da esquerda para a direita

# Imprimir informações sobre o arquivo e as coordenadas
print("Informações do arquivo .tif:")
print(f"Transformação: {transform}")
print(f"Limites: {bounds}")
print(f"Dimensões: {precip_data.shape}")

print("Primeiras 5 latitudes:", len(latitudes_GPM))
print("Primeiras 5 longitudes:",len(longitudes_GPM))

# Exibir a coordenada de latitude e longitude correspondentes ao primeiro valor de tempo, latitude e longitude
print("Primeiro valor de precipitação (na posição 0, 0):", precip_data[0, 0, 0])  # Tempo, Latitude, Longitude

Informações do arquivo .tif:
Transformação: | 0.10, 0.00,-85.00|
| 0.00,-0.10, 15.00|
| 0.00, 0.00, 1.00|
Limites: BoundingBox(left=-85.00000000000001, bottom=-60.000000000000014, right=-29.900000000000006, top=15.000000000000004)
Dimensões: (318, 750, 551)
Primeiras 5 latitudes: 750
Primeiras 5 longitudes: 551
Primeiro valor de precipitação (na posição 0, 0): 0.105000004


In [ ]:
import xarray as xr
# Criação do DataArray

anos = data_Tmax.astype('datetime64[Y]').astype(int) + 1970
ano = anos[0]
tmax = xr.Dataset(
    {
        "tmax": (["time", "lat", "lon"], tmax_raster - 273.15) 
    },
    coords={
        "time": data_Tmax,  # A nova dimensão representa os anos
        "lat": latitudes_tmax,
        "lon": longitudes_tmax
    }
)
    # Adicionando atributos ao dataset
tmax.attrs = {
    'title': 'tmax - Temperatura máxima média mensal',
    'units': '°C',
    'source': 'ERA-5 Land - ECMWF',
    'institution': 'IFUSP',
    'creator': 'Arthur Thomas Luz, Orientadores: Profa. Dra. Luciana Rizzo, Dr. Felipe Silva, Prof. Dr. Luiz Augusto Toledo Machado',
    'year_range': f'{anos.min()} - {anos.max()}',
    'geospatial_bounds': 'Oeste: -85°, Sul: -60°, Leste: -30°, Norte: 15°',
    'spatial_resolution': '0.1° x 0.1°',
    'temporal_resolution': 'Anual',
    'comments': 'Erro baseado na climatologia 1981-2010 comparada a 310 estações do INMET.',
    'history': 'Criado em 05/12/2024 com dados ERA-5 Land da Earth Engine.',
}

# Visualização dos dados
display(tmax)

#Salvando em NetCDF
tmax.to_netcdf(f"Dados_tmin_tmax/tmax_{ano}.nc")